In [1]:
import pandas as pd
import numpy as np

# Load Dataset

We load the transaction dataset containing banking transactions.

The dataset includes:
- Timestamp of transaction  
- Transaction amount  
- Transaction type (deposit/withdrawal)  

This will be used to analyze liquidity inflow and outflow patterns.

In [11]:
df = pd.read_csv("financial_fraud_detection_dataset(1).csv")

In [12]:
df = df[['timestamp', 'amount', 'transaction_type']]

In [13]:
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    errors='coerce',
    format='mixed'
)

df = df.dropna(subset=['timestamp'])

In [14]:
df.rename(columns={'timestamp': 'date'}, inplace=True)

In [15]:
df['deposit_amount'] = np.where(df['transaction_type'] == 'deposit', df['amount'], 0)
df['withdrawal_amount'] = np.where(df['transaction_type'] != 'deposit', df['amount'], 0)

In [16]:
df.columns

Index(['date', 'amount', 'transaction_type', 'deposit_amount',
       'withdrawal_amount'],
      dtype='str')

# Convert to Time-Series Data

We convert transaction-level data into time-series format.

Steps performed:
- Set datetime column as index  
- Resample data hourly  
- Aggregate deposits and withdrawals  
- Compute net flow (inflow - outflow)  

This helps analyze liquidity patterns over time.

In [17]:
df.set_index('date', inplace=True)

df_hourly = df.resample('h').sum()

df_hourly['net_flow'] = df_hourly['deposit_amount'] - df_hourly['withdrawal_amount']

# Time-Based Feature Engineering

In this step, we extract time-related features from the datetime index to capture patterns in liquidity behavior.

### ⏱ Time Features:
- **Hour** → identifies hourly patterns in transactions  
- **Day of Week** → captures weekday vs weekend behavior  
- **Weekend Indicator** → flags Saturday (5) and Sunday (6)  

### 🔁 Cyclical Encoding (Important Improvement):
Time features like hours are cyclical in nature (e.g., 23 → 0).

To preserve this relationship, we use sine and cosine transformations:

- `hour_sin`  
- `hour_cos`  

This helps the model understand the continuous nature of time instead of treating it as linear.

### 🎯 Purpose:
These features help the model learn:
- Daily transaction patterns  
- Weekly trends  
- Behavioral differences between weekdays and weekends  

This improves anomaly detection performance.

In [18]:
df_hourly['hour'] = df_hourly.index.hour
df_hourly['day_of_week'] = df_hourly.index.dayofweek
df_hourly['is_weekend'] = df_hourly['day_of_week'].isin([5,6]).astype(int)


df_hourly['hour_sin'] = np.sin(2 * np.pi * df_hourly['hour']/24)
df_hourly['hour_cos'] = np.cos(2 * np.pi * df_hourly['hour']/24)

# 📌 Step 6: Lag Features (Historical Dependency)

In this step, we create lag features to incorporate past information into the model.

### 🔁 Lag Features:
- **lag_1** → Net flow from the previous hour  
- **lag_24** → Net flow from the same hour on the previous day  

### 🧠 Why Lag Features?
Time-series data depends heavily on past behavior.  
Lag features help the model understand:

- Short-term trends (recent changes)  
- Daily patterns (same hour behavior across days)  

### 📊 Example:
If current time is 10 AM:
- `lag_1` → value at 9 AM  
- `lag_24` → value at 10 AM yesterday  

### 🎯 Purpose:
These features allow the model to:
- Capture temporal dependencies  
- Detect unusual deviations from normal patterns  
- Improve anomaly detection accuracy  

Lag features are essential for any time-series-based machine learning model.

In [19]:
df_hourly['lag_1'] = df_hourly['net_flow'].shift(1)
df_hourly['lag_24'] = df_hourly['net_flow'].shift(24)

In [20]:
df_hourly['rolling_mean_24'] = df_hourly['net_flow'].rolling(24).mean()
df_hourly['rolling_std_24'] = df_hourly['net_flow'].rolling(24).std()

In [21]:
df_hourly['net_flow_diff'] = df_hourly['net_flow'].diff()

In [22]:
df_hourly['pct_change'] = df_hourly['net_flow'].pct_change()

df_hourly.replace([np.inf, -np.inf], 0, inplace=True)

,amount,transaction_type,deposit_amount,withdrawal_amount,net_flow,hour,day_of_week,is_weekend,hour_sin,hour_cos,lag_1,lag_24,rolling_mean_24,rolling_std_24,net_flow_diff,pct_change
date,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,10526.06,transferdepositpaymentdepositdepositdeposittra...,8620.73,1905.33,6715.40,0,6,1,0.000000,1.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 01:00:00,17179.44,transferdepositpaymentpaymentpaymentdepositpay...,11253.96,5925.48,5328.48,1,6,1,0.258819,9.659258e-01,6715.40,NaN,NaN,NaN,-1386.92,-0.206528
2023-01-01 02:00:00,26571.44,paymenttransfertransferwithdrawaldeposittransf...,19905.36,6666.08,13239.28,2,6,1,0.500000,8.660254e-01,5328.48,NaN,NaN,NaN,7910.80,1.484626
2023-01-01 03:00:00,43071.17,transferwithdrawalwithdrawalwithdrawaldepositd...,33653.62,9417.55,24236.07,3,6,1,0.707107,7.071068e-01,13239.28,NaN,NaN,NaN,10996.79,0.830618
2023-01-01 04:00:00,44987.32,deposittransferpaymentpaymentdeposittransferde...,32357.57,12629.75,19727.82,4,6,1,0.866025,5.000000e-01,24236.07,NaN,NaN,NaN,-4508.25,-0.186014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-01-01 18:00:00,42598.30,transferpaymentdeposittransfertransferdepositw...,30151.58,12446.72,17704.86,18,0,0,-1.000000,-1.836970e-16,22352.95,70748.78,54764.923750,26577.672158,-4648.09,-0.207941
2024-01-01 19:00:00,30631.55,paymenttransferwithdrawalpaymentwithdrawaltran...,20942.61,9688.94,11253.67,19,0,0,-0.965926,2.588190e-01,17704.86,64492.99,52546.618750,27918.409073,-6451.19,-0.364374
2024-01-01 20:00:00,31781.53,depositwithdrawalwithdrawaltransferpaymenttran...,22384.81,9396.72,12988.09,20,0,0,-0.866025,5.000000e-01,11253.67,71021.22,50128.571667,28749.520574,1734.42,0.154120


In [41]:
inflation = pd.read_csv("inflation.csv", skiprows=4)
interest = pd.read_csv("interrest_rate.csv", skiprows=4)

inflation = inflation[inflation['Country Name'].str.contains('India', case=False, na=False)]
interest = interest[interest['Country Name'].str.contains('India', case=False, na=False)]

inflation = inflation.drop(columns=[
    'Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'
], errors='ignore')

interest = interest.drop(columns=[
    'Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'
], errors='ignore')

inflation = inflation.melt(var_name='year', value_name='inflation_rate')
interest = interest.melt(var_name='year', value_name='interest_rate')

inflation['year'] = pd.to_numeric(inflation['year'], errors='coerce')
interest['year'] = pd.to_numeric(interest['year'], errors='coerce')

inflation['inflation_rate'] = pd.to_numeric(inflation['inflation_rate'], errors='coerce')
interest['interest_rate'] = pd.to_numeric(interest['interest_rate'], errors='coerce')

inflation = inflation.dropna(subset=['year', 'inflation_rate'])
interest = interest.dropna(subset=['year', 'interest_rate'])

inflation['year'] = inflation['year'].astype(int)
interest['year'] = interest['year'].astype(int)

df_hourly['year'] = df_hourly.index.year

df_final = df_hourly.merge(inflation, on='year', how='left')
df_final = df_final.merge(interest, on='year', how='left')

df_final = df_final.ffill().bfill()

print(df_final[['year','inflation_rate','interest_rate']].head())
print(df_final.shape)

   year  inflation_rate  interest_rate
0  2023        5.649143            NaN
1  2023        5.649143            NaN
2  2023        5.649143            NaN
3  2023        5.649143            NaN
4  2023        5.649143            NaN
(8783, 19)


In [42]:
df_final['interest_rate'] = 6.5 + np.random.normal(0, 0.2, len(df_final))

In [43]:
df_final

,amount,transaction_type,deposit_amount,withdrawal_amount,net_flow,hour,day_of_week,is_weekend,hour_sin,hour_cos,lag_1,lag_24,rolling_mean_24,rolling_std_24,net_flow_diff,pct_change,year,inflation_rate,interest_rate
0,10526.06,transferdepositpaymentdepositdepositdeposittra...,8620.73,1905.33,6715.40,0,6,1,0.000000,1.000000e+00,6715.40,6715.40,43639.461250,26245.297650,-1386.92,-0.206528,2023,5.649143,6.247603
1,17179.44,transferdepositpaymentpaymentpaymentdepositpay...,11253.96,5925.48,5328.48,1,6,1,0.258819,9.659258e-01,6715.40,6715.40,43639.461250,26245.297650,-1386.92,-0.206528,2023,5.649143,6.448013
2,26571.44,paymenttransfertransferwithdrawaldeposittransf...,19905.36,6666.08,13239.28,2,6,1,0.500000,8.660254e-01,5328.48,6715.40,43639.461250,26245.297650,7910.80,1.484626,2023,5.649143,6.399645
3,43071.17,transferwithdrawalwithdrawalwithdrawaldepositd...,33653.62,9417.55,24236.07,3,6,1,0.707107,7.071068e-01,13239.28,6715.40,43639.461250,26245.297650,10996.79,0.830618,2023,5.649143,6.709553
4,44987.32,deposittransferpaymentpaymentdeposittransferde...,32357.57,12629.75,19727.82,4,6,1,0.866025,5.000000e-01,24236.07,6715.40,43639.461250,26245.297650,-4508.25,-0.186014,2023,5.649143,6.286047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8778,42598.30,transferpaymentdeposittransfertransferdepositw...,30151.58,12446.72,17704.86,18,0,0,-1.000000,-1.836970e-16,22352.95,70748.78,54764.923750,26577.672158,-4648.09,-0.207941,2024,4.953036,6.856139
8779,30631.55,paymenttransferwithdrawalpaymentwithdrawaltran...,20942.61,9688.94,11253.67,19,0,0,-0.965926,2.588190e-01,17704.86,64492.99,52546.618750,27918.409073,-6451.19,-0.364374,2024,4.953036,6.249922
8780,31781.53,depositwithdrawalwithdrawaltransferpaymenttran...,22384.81,9396.72,12988.09,20,0,0,-0.866025,5.000000e-01,11253.67,71021.22,50128.571667,28749.520574,1734.42,0.154120,2024,4.953036,6.606113
8781,24148.49,paymentdepositdepositwithdrawalwithdrawalwithd...,19942.63,4205.86,15736.77,21,0,0,-0.707107,7.071068e-01,12988.09,81678.35,47381.005833,28754.230041,2748.68,0.211631,2024,4.953036,6.474032


In [38]:
print("df shape:", df.shape)
print("df_hourly shape:", df_hourly.shape)

df shape: (5000000, 4)
df_hourly shape: (8783, 17)


Empty DataFrame
Columns: [year, interest_rate]
Index: []
(0, 2)


In [33]:
df_final = df_final.ffill().bfill()

df_final = df_final.dropna(subset=[
    'lag_1',
    'lag_24',
    'rolling_mean_24',
    'rolling_std_24'
])

In [45]:
print(df_final.shape)

(8783, 19)


In [46]:
df_final.isnull().sum()

amount               0
transaction_type     0
deposit_amount       0
withdrawal_amount    0
net_flow             0
hour                 0
day_of_week          0
is_weekend           0
hour_sin             0
hour_cos             0
lag_1                0
lag_24               0
rolling_mean_24      0
rolling_std_24       0
net_flow_diff        0
pct_change           0
year                 0
inflation_rate       0
interest_rate        0
dtype: int64

In [47]:
features = [
    'net_flow',
    'lag_1',
    'lag_24',
    'rolling_mean_24',
    'rolling_std_24',
    'pct_change',
    'net_flow_diff',
    'hour_sin',
    'hour_cos',
    'is_weekend',
    'inflation_rate',
    'interest_rate'
]

X = df_final[features]

In [48]:
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

In [49]:
print("df_final shape:", df_final.shape)
print("X shape:", X.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

df_final shape: (8783, 19)
X shape: (8783, 12)
X_train shape: (7026, 12)
X_test shape: (1757, 12)


In [50]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [51]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.01, random_state=42)

model.fit(X_train_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [52]:
train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)

train_pred = [1 if x == -1 else 0 for x in train_pred]
test_pred = [1 if x == -1 else 0 for x in test_pred]

In [53]:
df_final

,amount,transaction_type,deposit_amount,withdrawal_amount,net_flow,hour,day_of_week,is_weekend,hour_sin,hour_cos,lag_1,lag_24,rolling_mean_24,rolling_std_24,net_flow_diff,pct_change,year,inflation_rate,interest_rate
0,10526.06,transferdepositpaymentdepositdepositdeposittra...,8620.73,1905.33,6715.40,0,6,1,0.000000,1.000000e+00,6715.40,6715.40,43639.461250,26245.297650,-1386.92,-0.206528,2023,5.649143,6.247603
1,17179.44,transferdepositpaymentpaymentpaymentdepositpay...,11253.96,5925.48,5328.48,1,6,1,0.258819,9.659258e-01,6715.40,6715.40,43639.461250,26245.297650,-1386.92,-0.206528,2023,5.649143,6.448013
2,26571.44,paymenttransfertransferwithdrawaldeposittransf...,19905.36,6666.08,13239.28,2,6,1,0.500000,8.660254e-01,5328.48,6715.40,43639.461250,26245.297650,7910.80,1.484626,2023,5.649143,6.399645
3,43071.17,transferwithdrawalwithdrawalwithdrawaldepositd...,33653.62,9417.55,24236.07,3,6,1,0.707107,7.071068e-01,13239.28,6715.40,43639.461250,26245.297650,10996.79,0.830618,2023,5.649143,6.709553
4,44987.32,deposittransferpaymentpaymentdeposittransferde...,32357.57,12629.75,19727.82,4,6,1,0.866025,5.000000e-01,24236.07,6715.40,43639.461250,26245.297650,-4508.25,-0.186014,2023,5.649143,6.286047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8778,42598.30,transferpaymentdeposittransfertransferdepositw...,30151.58,12446.72,17704.86,18,0,0,-1.000000,-1.836970e-16,22352.95,70748.78,54764.923750,26577.672158,-4648.09,-0.207941,2024,4.953036,6.856139
8779,30631.55,paymenttransferwithdrawalpaymentwithdrawaltran...,20942.61,9688.94,11253.67,19,0,0,-0.965926,2.588190e-01,17704.86,64492.99,52546.618750,27918.409073,-6451.19,-0.364374,2024,4.953036,6.249922
8780,31781.53,depositwithdrawalwithdrawaltransferpaymenttran...,22384.81,9396.72,12988.09,20,0,0,-0.866025,5.000000e-01,11253.67,71021.22,50128.571667,28749.520574,1734.42,0.154120,2024,4.953036,6.606113
8781,24148.49,paymentdepositdepositwithdrawalwithdrawalwithd...,19942.63,4205.86,15736.77,21,0,0,-0.707107,7.071068e-01,12988.09,81678.35,47381.005833,28754.230041,2748.68,0.211631,2024,4.953036,6.474032


In [54]:
df_final.to_csv("df_final.csv")